In [37]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import pandas as pd
from sqlalchemy import create_engine
import time
import locale

# Configurar la localización a español
locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')

'es_ES.UTF-8'

In [38]:
# Configuración de la conexión
server = '10.0.0.129'
username = 'sa'
password = 'Essalud23**'
driver = 'ODBC Driver 17 for SQL Server'
database = 'DB_TRAMITE_DOCUMENTARIO'  # Reemplaza con el nombre de tu base de datos
# Cadena de conexión
connection_string = f'mssql+pyodbc://{username}:{password}@{server}/{database}?driver={driver}'

In [39]:
try:
    # Establecer la conexión con SQLAlchemy
    engine = create_engine(connection_string)

    # Construir la consulta SQL
    query = """
    SELECT d.ID_DOCUMENTO, d.ID_TIPO_DOCUMENTO, d.ID_ESTADO_DOCUMENTO, d.AUDITMOD ,d.hoja_tramite  ,d.ASUNTO, d.FECHA_RECEPCION, p.ID, tp.DESCRIPCION , p.RAZON_SOCIAL, p.NRO_DOCUMENTO,p.NOMBRES, p.APELLIDOS, p.EMAIL 
FROM DB_TRAMITE_DOCUMENTARIO.web_tramite.DOCUMENTO d
LEFT OUTER JOIN DB_GENERAL.dbo.PERSONA p
ON d.ID_PERSONA = p.ID
LEFT OUTER JOIN DB_GENERAL.dbo.TIPO_PERSONA tp 
ON p.ID_TIPO_PERSONA = tp.ID_TIPO_PERSONA 
WHERE d.ID_DOCUMENTO IN (SELECT DISTINCT ID_DOCUMENTO
                         FROM DB_TRAMITE_DOCUMENTARIO.web_tramite.MOVIMIENTO_DOCUMENTO md)
AND d.ID_TIPO_DOCUMENTO = 1
AND d.AUDITMOD >= CONVERT(DATETIME, '2024-01-02', 120)
AND d.AUDITMOD < CONVERT(DATETIME, '2024-02-13', 120)
    """

    # Leer los resultados en un DataFrame de Pandas
    df = pd.read_sql(query, engine)

except Exception as e:
    print(f'Error de conexión: {e}')

finally:
    # No es necesario cerrar la conexión con SQLAlchemy
    pass

In [40]:
# Crear la columna 'usuario' basada en la lógica dada
df['usuario'] = df.apply(lambda row: row['RAZON_SOCIAL'] if row['DESCRIPCION'] == 'JURIDICA' else f"{row['NOMBRES']} {row['APELLIDOS']}", axis=1)

In [41]:
df_final = pd.DataFrame()
df_final['usuario'] = df['usuario']
df_final['nro_dni'] = df['NRO_DOCUMENTO']
df_final['nro_ht'] = df['hoja_tramite']
df_final['correo'] = df['EMAIL']
df_final['fecha'] = df['AUDITMOD']

In [42]:
df_final = df_final.dropna(subset=['correo'])
df_final = df_final.dropna(subset=['nro_dni'])
df_final = df_final.dropna(subset=['nro_ht'])
df_final['fecha'] = df_final['fecha'].dt.strftime('%d%b%Y').str.capitalize()

In [43]:
df_final['cadena'] = df_final['nro_dni']+'-'+df_final['nro_ht']

In [44]:
df_final

,usuario,nro_dni,nro_ht,correo,fecha,cadena
4,REDESS CHUCUITO,20363734074,00000006-2024,REDCHUCUITOJULI.DIRECCION@GMAIL.COM,03ene2024,20363734074-00000006-2024
5,REDESS CHUCUITO,20363734074,00000007-2024,REDCHUCUITOJULI.DIRECCION@GMAIL.COM,03ene2024,20363734074-00000007-2024
6,REDESS CHUCUITO,20363734074,00000008-2024,REDCHUCUITOJULI.DIRECCION@GMAIL.COM,03ene2024,20363734074-00000008-2024
7,REDESS CHUCUITO,20363734074,00000009-2024,REDCHUCUITOJULI.DIRECCION@GMAIL.COM,03ene2024,20363734074-00000009-2024
8,VASQUEZ BERNAL SOLEDAD KARINA,10442610997,00000010-2024,KARINAVIB44261099@GMAIL.COM,03ene2024,10442610997-00000010-2024
...,...,...,...,...,...,...
11099,SEGALF S.A.C.,20611740761,00011284-2024,segalfsac@gmail.com,12feb2024,20611740761-00011284-2024
11205,ENMA MERCEDES LEDESMA FLORES,43506462,00011390-2024,mercedesledesmaflores@gmail.com,12feb2024,43506462-00011390-2024
11207,RICARDO EMILIO LOPEZ DAVALOS,07268697,00011392-2024,riloda1973@gmail.com,12feb2024,07268697-00011392-2024
11212,DORIS ROSSANA VELASQUEZ OLORTEGUI,32102234,00011397-2024,rossanavelasquez8@gmail.com,12feb2024,32102234-00011397-2024


In [45]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1856 entries, 4 to 11213
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   usuario  1856 non-null   object
 1   nro_dni  1856 non-null   object
 2   nro_ht   1856 non-null   object
 3   correo   1856 non-null   object
 4   fecha    1856 non-null   object
 5   cadena   1856 non-null   object
dtypes: object(6)
memory usage: 101.5+ KB


In [30]:
#ruta_y_nombre_archivo = '/home/ugadingenieria01/Documentos/GCTIC/DW_GCTIC/HT.csv'

# Guardar el DataFrame en el archivo CSV
#df_final.to_csv(ruta_y_nombre_archivo, index=False, encoding='utf-8')

In [31]:
#df_final = df_final.tail(2)

In [32]:
#df_final

In [33]:
#df_prueba = df_final[df_final['nro_dni'] == '72299187']

In [34]:
#df_prueba

In [46]:
def enviar_correo(asunto, destinatario, usuario, nro_ht, nro_dni, fecha):
    # Configuración de correo
    mail_settings = {
        "user": "std@essalud.gob.pe",
        "password": "essaludc24",
        "host": "wiracocha.essalud",
        "port": 25,
        "ssl": False
    }

    # Leer la plantilla HTML desde el archivo
    with open('./NotificacionMPE.html', 'r') as file:
        plantilla_html = file.read()

    # Reemplazar las variables en la plantilla
    plantilla_html = plantilla_html.replace('{{usuario}}', usuario)
    plantilla_html = plantilla_html.replace('{{nro_ht}}', nro_ht)
    plantilla_html = plantilla_html.replace('{{nro_dni}}', nro_dni)
    plantilla_html = plantilla_html.replace('{{fecha}}', fecha)

    # Crear mensaje
    mensaje = MIMEMultipart()
    mensaje['From'] = mail_settings["user"]
    mensaje['To'] = destinatario
    mensaje['Subject'] = asunto

    # Cuerpo del mensaje (parte HTML)
    cuerpo_html = MIMEText(plantilla_html, 'html')
    mensaje.attach(cuerpo_html)

    # Conexión SMTP
    try:
        servidor_smtp = smtplib.SMTP(mail_settings["host"], mail_settings["port"])

        # Envío del correo
        servidor_smtp.sendmail(mail_settings["user"], destinatario, mensaje.as_string())

        # Cierre de la conexión
        servidor_smtp.quit()
        print("Correo enviado exitosamente.")
    except Exception as e:
        print(f"Error al enviar el correo: {str(e)} hoja_tramite:{nro_ht}")

for index, row in df_final.iterrows():
    asunto = "SGD - Notificación automática para seguimiento de trámite"
    destinatario = row['correo']  # Puedes utilizar row['columna'] para obtener valores de la fila actual
    usuario = row['usuario']
    nro_ht = row['nro_ht']
    nro_dni = row['nro_dni']
    fecha = row['fecha']
    enviar_correo(asunto, destinatario, usuario, nro_ht, nro_dni, fecha)
    time.sleep(0.5)

Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado exitosamente.
Correo enviado